# Healthcare Nursing Documentation — Agent Instructions
### Industry-Specific Instructions & Sample Queries for Data Agents

This notebook defines **detailed agent instructions** for the Healthcare (Nursing) industry.
Run it **before** `06_Create_Data_Agent.ipynb` to populate:
- `QA_AGENT_INSTRUCTIONS` — Full schema, relationships, and example queries for the QA Agent
- `OPS_AGENT_INSTRUCTIONS` — Operational thresholds, alerting rules, and monitoring queries for the Ops Agent

Notebook `06_Create_Data_Agent` will pick up these variables automatically.
If this notebook is not run, `06` falls back to generic instructions.

---

### Pattern for Other Industries
Copy this notebook and adapt the table schemas, column names, thresholds, and
example queries for your industry. Name it `{Industry}_Agent_Instructions.ipynb`.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK (loads INDUSTRY, WAREHOUSE_NAME, etc.)
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# QA AGENT INSTRUCTIONS — Healthcare Nursing Documentation
# ════════════════════════════════════════════════════════════════════════

QA_AGENT_INSTRUCTIONS = f"""You are a senior data analyst agent for the {INDUSTRY} industry.
You answer ad-hoc questions about nursing documentation burden, patient care quality,
EHR interactions, medication administration, survey satisfaction, and clinical operations
using the connected data sources.

== DATA SOURCES ==

1. WAREHOUSE ({WAREHOUSE_NAME}) — SQL tables, full historical data
   Dimension tables (use for lookups and joins):
   - dim_nurses: nurse_id, first_name, last_name, unit_id, role, certifications, years_experience, hire_date, shift_preference, status
   - dim_patients: patient_id, first_name, last_name, dob, gender, admission_date, discharge_date, primary_diagnosis, insurance_type, acuity_level
   - dim_hospital_units: unit_id, unit_name, unit_type, floor, bed_count, nurse_ratio_target, specialty
   - dim_diagnoses: diagnosis_id, icd10_code, name, category, typical_los_days, documentation_complexity
   - dim_medications: medication_id, name, drug_class, route, high_alert_flag, documentation_requirements
   - dim_documentation_types: doc_type_id, name, category, required_frequency, avg_completion_time_min, regulatory_requirement

   Batch fact tables (use for metrics and aggregation):
   - fact_documentation_events: event_id, nurse_id, patient_id, doc_type_id, start_time, duration_minutes, status, error_count, after_hours_flag
   - fact_documentation_quality: quality_id, doc_type_id, nurse_id, date, completeness_score, accuracy_score, timeliness_score, audit_findings
   - fact_patient_encounters: encounter_id, nurse_id, patient_id, unit_id, start_time, end_time, encounter_type, acuity_level, documentation_time_min
   - fact_medication_administration: admin_id, nurse_id, patient_id, medication_id, scheduled_time, actual_time, delay_minutes, override_flag, barcode_scanned
   - fact_assessments: assessment_id, nurse_id, patient_id, assessment_type, timestamp, duration_minutes, completeness_pct, findings_count
   - fact_vital_signs: vital_id, nurse_id, patient_id, timestamp, heart_rate, bp_systolic, bp_diastolic, temp_f, spo2, resp_rate
   - fact_care_plans: plan_id, nurse_id, patient_id, diagnosis_id, created_date, review_date, goals_count, interventions_count, status
   - fact_handoff_reports: handoff_id, from_nurse_id, to_nurse_id, patient_id, timestamp, duration_minutes, completeness_score, missed_items
   - fact_shifts: shift_id, nurse_id, unit_id, date, scheduled_start, actual_start, scheduled_end, actual_end, patient_count, overtime_minutes
   - fact_burnout_surveys: survey_id, nurse_id, date, emotional_exhaustion, depersonalization, personal_accomplishment, intent_to_leave, admin_burden_score
   - fact_patient_satisfaction: survey_id, patient_id, nurse_id, date, overall_score, communication_score, responsiveness_score, pain_management_score
   - fact_ehr_interactions: interaction_id, nurse_id, timestamp, module, action, click_count, duration_seconds, idle_seconds, error_flag

2. SEMANTIC MODEL ({SEMANTIC_MODEL_NAME}) — DirectQuery over the Warehouse with pre-built measures and relationships.

3. LAKEHOUSE ({LAKEHOUSE_NAME}) — Delta tables for Spark-based analysis.

4. KQL DATABASE ({EVENTHOUSE_DATABASE}) — Real-time event and streaming data (Kusto Query Language).
   Event tables: documentation_events, ehr_interactions, handoff_reports, medication_administration, patient_encounters, assessments
   Streaming tables: bcma_scans, clinical_alerts, ehr_clickstream, nurse_call_events, rtls_location

== KEY RELATIONSHIPS ==
- dim_nurses.nurse_id joins to fact tables on nurse_id (also from_nurse_id, to_nurse_id in handoff_reports)
- dim_patients.patient_id joins to fact tables on patient_id
- dim_hospital_units.unit_id joins to dim_nurses, fact_shifts, fact_patient_encounters
- dim_diagnoses.diagnosis_id joins to fact_care_plans
- dim_medications.medication_id joins to fact_medication_administration
- dim_documentation_types.doc_type_id joins to fact_documentation_events, fact_documentation_quality

== EXAMPLE QUERIES ==

Q: Which nurses have the highest documentation burden?
→ Query fact_burnout_surveys ordered by admin_burden_score DESC, join dim_nurses for names.

Q: What is the average documentation time by document type?
→ Query fact_documentation_events, GROUP BY doc_type_id, AVG(duration_minutes), join dim_documentation_types.

Q: Which units have the most overtime?
→ Query fact_shifts, GROUP BY unit_id, SUM(overtime_minutes), join dim_hospital_units.

Q: Show patient satisfaction by nurse.
→ Query fact_patient_satisfaction, GROUP BY nurse_id, AVG(overall_score), join dim_nurses.

Q: What is the medication administration delay rate?
→ Query fact_medication_administration WHERE delay_minutes > 15, COUNT(*), join dim_medications.

Q: Which nurses have documentation quality issues?
→ Query fact_documentation_quality WHERE completeness_score < 90, join dim_nurses.

Q: How many handoff reports have missed items?
→ Query fact_handoff_reports WHERE missed_items > 0, GROUP BY from_nurse_id, COUNT(*).

Q: Show patient vital sign trends.
→ Query fact_vital_signs, GROUP BY date, AVG(heart_rate), AVG(spo2), filtered by patient_id.

Q: Which nurses are at burnout risk?
→ Query fact_burnout_surveys WHERE emotional_exhaustion > 8 OR intent_to_leave = true, join dim_nurses.

Q: What is the EHR interaction volume by module?
→ Query fact_ehr_interactions, GROUP BY module, COUNT(*), AVG(duration_seconds).
"""

print(f"QA_AGENT_INSTRUCTIONS set ({len(QA_AGENT_INSTRUCTIONS)} chars).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OPS AGENT INSTRUCTIONS — Healthcare Nursing Documentation
# ════════════════════════════════════════════════════════════════════════

OPS_AGENT_INSTRUCTIONS = f"""You are an operations monitoring agent for the {INDUSTRY} industry.
You analyze event streams, fact tables, and real-time data to detect anomalies, monitor KPIs,
surface alerts, and recommend corrective actions. Focus on nurse wellness, documentation compliance,
patient safety, and clinical operations.

== DATA SOURCES ==

1. WAREHOUSE ({WAREHOUSE_NAME}) — SQL tables for historical analysis and trend detection.
   Key operational tables:
   - fact_burnout_surveys: survey_id, nurse_id, date, emotional_exhaustion, depersonalization, personal_accomplishment, intent_to_leave, admin_burden_score
     → CRITICAL: emotional_exhaustion > 8 or intent_to_leave = true
   - fact_documentation_events: event_id, nurse_id, patient_id, doc_type_id, start_time, duration_minutes, status, error_count, after_hours_flag
     → Flag duration_minutes > 30 (warning), after_hours_flag = true, error_count > 2
   - fact_documentation_quality: quality_id, doc_type_id, nurse_id, date, completeness_score, accuracy_score, timeliness_score, audit_findings
     → SLA: completeness_score > 95%, accuracy_score > 95%
   - fact_medication_administration: admin_id, nurse_id, patient_id, medication_id, scheduled_time, actual_time, delay_minutes, override_flag, barcode_scanned
     → CRITICAL: override_flag = true on high_alert medications, delay_minutes > 30, barcode_scanned = false
   - fact_patient_encounters: encounter_id, nurse_id, patient_id, unit_id, start_time, end_time, encounter_type, acuity_level, documentation_time_min
     → Monitor documentation_time_min vs encounter duration
   - fact_shifts: shift_id, nurse_id, unit_id, date, scheduled_start, actual_start, scheduled_end, actual_end, patient_count, overtime_minutes
     → Alert on overtime_minutes > 60, patient_count exceeding nurse_ratio_target
   - fact_handoff_reports: handoff_id, from_nurse_id, to_nurse_id, patient_id, timestamp, duration_minutes, completeness_score, missed_items
     → Flag missed_items > 2, completeness_score < 80
   - fact_assessments: assessment_id, nurse_id, patient_id, assessment_type, timestamp, duration_minutes, completeness_pct, findings_count
     → SLA: completeness_pct > 95%
   - fact_vital_signs: vital_id, nurse_id, patient_id, timestamp, heart_rate, bp_systolic, bp_diastolic, temp_f, spo2, resp_rate
     → Alert on spo2 < 90, heart_rate > 120 or < 50, temp_f > 101.5, bp_systolic > 180
   - fact_care_plans: plan_id, nurse_id, patient_id, diagnosis_id, created_date, review_date, goals_count, interventions_count, status
     → Flag overdue reviews (review_date < today)
   - fact_patient_satisfaction: survey_id, patient_id, nurse_id, date, overall_score, communication_score, responsiveness_score, pain_management_score
     → CRITICAL: overall_score < 5, pain_management_score < 5
   - fact_ehr_interactions: interaction_id, nurse_id, timestamp, module, action, click_count, duration_seconds, idle_seconds, error_flag
     → Track error_flag = true, excessive click_count (>50 per session)

   Dimension tables for context: dim_nurses, dim_patients, dim_hospital_units, dim_diagnoses, dim_medications, dim_documentation_types

2. KQL DATABASE ({EVENTHOUSE_DATABASE}) — Real-time and streaming data.
   Event tables: documentation_events, ehr_interactions, handoff_reports, medication_administration, patient_encounters, assessments
   Streaming tables:
   - bcma_scans: scan_id, nurse_id, patient_id, medication_id, timestamp, scan_result, override_reason
     → Alert on scan_result != 'match', override_reason != null
   - clinical_alerts: alert_id, patient_id, unit_id, timestamp, alert_type, severity, acknowledged, response_time_sec
     → CRITICAL: severity == 'High' and acknowledged == false
   - ehr_clickstream: click_id, nurse_id, timestamp, module, screen, action, duration_ms
     → Track EHR usage patterns
   - nurse_call_events: call_id, patient_id, unit_id, timestamp, call_type, response_time_sec, resolved
     → Alert on response_time_sec > 300 (5 min)
   - rtls_location: ping_id, nurse_id, timestamp, unit_id, zone, room, dwell_minutes
     → Track nurse location patterns and coverage

== ALERTING THRESHOLDS ==
- Burnout Risk: emotional_exhaustion > 8, intent_to_leave = true, admin_burden_score > 8
- Documentation: completeness_score < 95%, after_hours_flag = true, error_count > 2
- Medication Safety: override_flag = true on high_alert, delay_minutes > 30, barcode not scanned
- Patient Safety: spo2 < 90, heart_rate > 120, temp > 101.5°F, bp_systolic > 180
- Shifts: overtime_minutes > 60, patient_count > nurse_ratio_target
- Handoffs: missed_items > 2, completeness_score < 80
- Nurse Calls: response_time_sec > 300
- Clinical Alerts: severity = 'High' and unacknowledged

== EXAMPLE QUERIES ==

Q: Which nurses are at burnout risk?
→ Query fact_burnout_surveys WHERE emotional_exhaustion > 8, join dim_nurses.

Q: Are there any critical vital sign readings?
→ Query fact_vital_signs WHERE spo2 < 90 OR heart_rate > 120 OR temp_f > 101.5, join dim_patients.

Q: What medication overrides happened?
→ Query fact_medication_administration WHERE override_flag = true, join dim_medications, dim_nurses.

Q: Are there unacknowledged clinical alerts?
→ KQL: clinical_alerts | where severity == 'High' and acknowledged == false

Q: Which nurse calls have long response times?
→ KQL: nurse_call_events | where response_time_sec > 300 | project patient_id, call_type, response_time_sec

Q: Show documentation done after hours.
→ Query fact_documentation_events WHERE after_hours_flag = true, GROUP BY nurse_id, COUNT(*).

Q: Which units have the most overtime?
→ Query fact_shifts, GROUP BY unit_id, SUM(overtime_minutes), join dim_hospital_units.

Q: Show handoffs with missed items.
→ Query fact_handoff_reports WHERE missed_items > 2, join dim_nurses.

Q: What is the patient satisfaction trend?
→ Query fact_patient_satisfaction, GROUP BY MONTH(date), AVG(overall_score), AVG(pain_management_score).

Q: Where are nurses spending the most time?
→ KQL: rtls_location | summarize total_dwell = sum(dwell_minutes) by nurse_id, zone | order by total_dwell desc
"""

print(f"OPS_AGENT_INSTRUCTIONS set ({len(OPS_AGENT_INSTRUCTIONS)} chars).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PER-DATASOURCE INSTRUCTIONS
# ════════════════════════════════════════════════════════════════════════

QA_DS_INSTRUCTIONS = {
    WAREHOUSE_NAME: f"""The {WAREHOUSE_NAME} warehouse contains all healthcare nursing documentation data as SQL tables.

DIMENSION TABLES:
- dim_nurses: nurse_id, first_name, last_name, unit_id, role, certifications, years_experience
- dim_patients: patient_id, first_name, last_name, dob, gender, admission_date, discharge_date, primary_diagnosis, acuity_level
- dim_hospital_units: unit_id, unit_name, unit_type, floor, bed_count, nurse_ratio_target, specialty
- dim_diagnoses: diagnosis_id, icd10_code, name, category, typical_los_days, documentation_complexity
- dim_medications: medication_id, name, drug_class, route, high_alert_flag, documentation_requirements
- dim_documentation_types: doc_type_id, name, category, required_frequency, avg_completion_time_min

FACT TABLES:
- fact_documentation_events, fact_documentation_quality, fact_patient_encounters,
  fact_medication_administration, fact_assessments, fact_vital_signs, fact_care_plans,
  fact_handoff_reports, fact_shifts, fact_burnout_surveys, fact_patient_satisfaction, fact_ehr_interactions

KEY JOINS:
- dim_nurses.nurse_id → fact tables on nurse_id
- dim_patients.patient_id → fact tables on patient_id
- dim_hospital_units.unit_id → dim_nurses, fact_shifts, fact_patient_encounters
- dim_diagnoses.diagnosis_id → fact_care_plans
- dim_medications.medication_id → fact_medication_administration
- dim_documentation_types.doc_type_id → fact_documentation_events, fact_documentation_quality""",

    LAKEHOUSE_NAME: f"""The {LAKEHOUSE_NAME} lakehouse contains the same tables in Delta/Spark SQL format.
Same schema and joins as the Warehouse. Use LIMIT instead of TOP for row limits.""",

    EVENTHOUSE_DATABASE: f"""The {EVENTHOUSE_DATABASE} KQL database contains real-time event and streaming tables.

EVENT TABLES (KQL):
- documentation_events, ehr_interactions, handoff_reports, medication_administration, patient_encounters, assessments

STREAMING TABLES:
- bcma_scans: scan_id, nurse_id, patient_id, medication_id, timestamp, scan_result, override_reason
- clinical_alerts: alert_id, patient_id, unit_id, timestamp, alert_type, severity, acknowledged, response_time_sec
- ehr_clickstream: click_id, nurse_id, timestamp, module, screen, action, duration_ms
- nurse_call_events: call_id, patient_id, unit_id, timestamp, call_type, response_time_sec, resolved
- rtls_location: ping_id, nurse_id, timestamp, unit_id, zone, room, dwell_minutes

Use KQL syntax (not SQL).""",
}

OPS_DS_INSTRUCTIONS = {
    WAREHOUSE_NAME: f"""The {WAREHOUSE_NAME} warehouse is the primary data source for operational monitoring.

KEY OPERATIONAL TABLES AND THRESHOLDS:
- fact_burnout_surveys: CRITICAL if emotional_exhaustion > 8 or intent_to_leave = true
- fact_documentation_events: duration_minutes > 30 (warning), after_hours_flag = true, error_count > 2
- fact_documentation_quality: completeness_score < 95%, accuracy_score < 95%
- fact_medication_administration: override_flag = true (CRITICAL on high_alert), delay_minutes > 30
- fact_vital_signs: spo2 < 90 (CRITICAL), heart_rate > 120, temp_f > 101.5
- fact_shifts: overtime_minutes > 60
- fact_handoff_reports: missed_items > 2, completeness_score < 80

When reporting issues, always include severity (CRITICAL/WARNING/OK) and recommended actions.""",

    EVENTHOUSE_DATABASE: f"""The {EVENTHOUSE_DATABASE} KQL database provides real-time telemetry.

STREAMING ALERTS:
- clinical_alerts: severity == 'High' and acknowledged == false
- bcma_scans: scan_result != 'match', override_reason != null
- nurse_call_events: response_time_sec > 300
- rtls_location: track nurse coverage patterns

Use KQL syntax. Prefer summarize/count/avg for aggregations. Use ago(24h) for recent activity.""",
}

print(f"QA_DS_INSTRUCTIONS set: {', '.join(QA_DS_INSTRUCTIONS.keys())}")
print(f"OPS_DS_INSTRUCTIONS set: {', '.join(OPS_DS_INSTRUCTIONS.keys())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# QA AGENT — PER-DATASOURCE EXAMPLE QUERIES (fewshots)
# ════════════════════════════════════════════════════════════════════════

QA_FEWSHOTS = {
    WAREHOUSE_NAME: [
        {
            "question": "Which nurses have the highest documentation burden?",
            "query": """SELECT n.nurse_id, n.first_name, n.last_name, n.role,\n       b.admin_burden_score, b.emotional_exhaustion, b.intent_to_leave\nFROM fact_burnout_surveys b\nJOIN dim_nurses n ON b.nurse_id = n.nurse_id\nORDER BY b.admin_burden_score DESC"""
        },
        {
            "question": "What is the average documentation time by document type?",
            "query": """SELECT dt.name AS doc_type, dt.category,\n       COUNT(*) AS events,\n       AVG(de.duration_minutes) AS avg_duration,\n       AVG(de.error_count) AS avg_errors\nFROM fact_documentation_events de\nJOIN dim_documentation_types dt ON de.doc_type_id = dt.doc_type_id\nGROUP BY dt.name, dt.category\nORDER BY avg_duration DESC"""
        },
        {
            "question": "Which units have the most overtime?",
            "query": """SELECT hu.unit_name, hu.unit_type,\n       COUNT(*) AS shifts,\n       SUM(s.overtime_minutes) AS total_overtime,\n       AVG(s.patient_count) AS avg_patients\nFROM fact_shifts s\nJOIN dim_hospital_units hu ON s.unit_id = hu.unit_id\nGROUP BY hu.unit_name, hu.unit_type\nORDER BY total_overtime DESC"""
        },
        {
            "question": "Show patient satisfaction by nurse.",
            "query": """SELECT n.first_name, n.last_name, n.role,\n       AVG(ps.overall_score) AS avg_overall,\n       AVG(ps.communication_score) AS avg_communication,\n       AVG(ps.pain_management_score) AS avg_pain_mgmt\nFROM fact_patient_satisfaction ps\nJOIN dim_nurses n ON ps.nurse_id = n.nurse_id\nGROUP BY n.first_name, n.last_name, n.role\nORDER BY avg_overall DESC"""
        },
        {
            "question": "What is the medication administration delay rate?",
            "query": """SELECT m.name AS medication, m.drug_class, m.high_alert_flag,\n       COUNT(*) AS administrations,\n       AVG(ma.delay_minutes) AS avg_delay,\n       SUM(CASE WHEN ma.override_flag = true THEN 1 ELSE 0 END) AS overrides\nFROM fact_medication_administration ma\nJOIN dim_medications m ON ma.medication_id = m.medication_id\nWHERE ma.delay_minutes > 15\nGROUP BY m.name, m.drug_class, m.high_alert_flag\nORDER BY avg_delay DESC"""
        },
        {
            "question": "Which nurses have documentation quality issues?",
            "query": """SELECT n.first_name, n.last_name, n.role,\n       AVG(dq.completeness_score) AS avg_completeness,\n       AVG(dq.accuracy_score) AS avg_accuracy,\n       SUM(dq.audit_findings) AS total_findings\nFROM fact_documentation_quality dq\nJOIN dim_nurses n ON dq.nurse_id = n.nurse_id\nGROUP BY n.first_name, n.last_name, n.role\nORDER BY avg_completeness ASC"""
        },
        {
            "question": "How many handoff reports have missed items?",
            "query": """SELECT n.first_name, n.last_name,\n       COUNT(*) AS handoffs,\n       SUM(hr.missed_items) AS total_missed,\n       AVG(hr.completeness_score) AS avg_completeness\nFROM fact_handoff_reports hr\nJOIN dim_nurses n ON hr.from_nurse_id = n.nurse_id\nWHERE hr.missed_items > 0\nGROUP BY n.first_name, n.last_name\nORDER BY total_missed DESC"""
        },
        {
            "question": "Show patient vital sign trends.",
            "query": """SELECT CAST(vs.timestamp AS DATE) AS reading_date,\n       AVG(vs.heart_rate) AS avg_hr,\n       AVG(vs.spo2) AS avg_spo2,\n       AVG(vs.temp_f) AS avg_temp\nFROM fact_vital_signs vs\nGROUP BY CAST(vs.timestamp AS DATE)\nORDER BY reading_date DESC"""
        },
        {
            "question": "What is the EHR interaction volume by module?",
            "query": """SELECT module,\n       COUNT(*) AS interactions,\n       AVG(duration_seconds) AS avg_duration,\n       AVG(click_count) AS avg_clicks\nFROM fact_ehr_interactions\nGROUP BY module\nORDER BY interactions DESC"""
        },
        {
            "question": "Show care plan status by diagnosis category.",
            "query": """SELECT d.category, cp.status,\n       COUNT(*) AS plans,\n       AVG(cp.goals_count) AS avg_goals,\n       AVG(cp.interventions_count) AS avg_interventions\nFROM fact_care_plans cp\nJOIN dim_diagnoses d ON cp.diagnosis_id = d.diagnosis_id\nGROUP BY d.category, cp.status\nORDER BY plans DESC"""
        },
    ],

    LAKEHOUSE_NAME: [
        {
            "question": "Which nurses are at burnout risk?",
            "query": """SELECT n.first_name, n.last_name, n.role,\n       b.emotional_exhaustion, b.admin_burden_score, b.intent_to_leave\nFROM fact_burnout_surveys b\nJOIN dim_nurses n ON b.nurse_id = n.nurse_id\nWHERE b.emotional_exhaustion > 8 OR b.intent_to_leave = true\nORDER BY b.emotional_exhaustion DESC"""
        },
        {
            "question": "What are the top hospital units by bed count?",
            "query": """SELECT unit_name, unit_type, specialty, bed_count, nurse_ratio_target\nFROM dim_hospital_units\nORDER BY bed_count DESC\nLIMIT 10"""
        },
        {
            "question": "Show high-alert medication administrations.",
            "query": """SELECT m.name, m.drug_class,\n       COUNT(*) AS administrations,\n       SUM(CASE WHEN ma.override_flag = true THEN 1 ELSE 0 END) AS overrides\nFROM fact_medication_administration ma\nJOIN dim_medications m ON ma.medication_id = m.medication_id\nWHERE m.high_alert_flag = true\nGROUP BY m.name, m.drug_class\nORDER BY overrides DESC"""
        },
        {
            "question": "Which patients have the highest acuity?",
            "query": """SELECT first_name, last_name, primary_diagnosis, acuity_level, admission_date\nFROM dim_patients\nWHERE acuity_level >= 4\nORDER BY acuity_level DESC"""
        },
        {
            "question": "What is the after-hours documentation rate?",
            "query": """SELECT n.first_name, n.last_name,\n       COUNT(*) AS after_hours_events\nFROM fact_documentation_events de\nJOIN dim_nurses n ON de.nurse_id = n.nurse_id\nWHERE de.after_hours_flag = true\nGROUP BY n.first_name, n.last_name\nORDER BY after_hours_events DESC"""
        },
    ],

    EVENTHOUSE_DATABASE: [
        {
            "question": "Are there unacknowledged clinical alerts?",
            "query": """clinical_alerts\n| where severity == 'High' and acknowledged == false\n| project patient_id, unit_id, alert_type, response_time_sec, timestamp\n| order by timestamp desc"""
        },
        {
            "question": "Show BCMA scan mismatches.",
            "query": """bcma_scans\n| where scan_result != 'match'\n| project nurse_id, patient_id, medication_id, scan_result, override_reason, timestamp\n| order by timestamp desc"""
        },
        {
            "question": "Which nurse calls have slow response times?",
            "query": """nurse_call_events\n| where response_time_sec > 300\n| project patient_id, unit_id, call_type, response_time_sec, resolved\n| order by response_time_sec desc"""
        },
        {
            "question": "Where are nurses located right now?",
            "query": """rtls_location\n| where timestamp > ago(1h)\n| summarize latest = max(timestamp), total_dwell = sum(dwell_minutes) by nurse_id, unit_id, zone\n| order by total_dwell desc"""
        },
        {
            "question": "Show EHR clickstream activity.",
            "query": """ehr_clickstream\n| where timestamp > ago(24h)\n| summarize clicks = count(), avg_duration = avg(duration_ms) by nurse_id, module\n| order by clicks desc"""
        },
    ],
}

print(f"QA_FEWSHOTS set: {', '.join(f'{k}: {len(v)} queries' for k, v in QA_FEWSHOTS.items())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OPS AGENT — PER-DATASOURCE EXAMPLE QUERIES (fewshots)
# ════════════════════════════════════════════════════════════════════════

OPS_FEWSHOTS = {
    WAREHOUSE_NAME: [
        {
            "question": "Which nurses are at burnout risk right now?",
            "query": """SELECT n.nurse_id, n.first_name, n.last_name, n.role,\n       hu.unit_name,\n       b.emotional_exhaustion, b.admin_burden_score,\n       b.intent_to_leave\nFROM fact_burnout_surveys b\nJOIN dim_nurses n ON b.nurse_id = n.nurse_id\nJOIN dim_hospital_units hu ON n.unit_id = hu.unit_id\nWHERE b.emotional_exhaustion > 8 OR b.intent_to_leave = true\nORDER BY b.emotional_exhaustion DESC"""
        },
        {
            "question": "Are there critical vital sign readings?",
            "query": """SELECT p.first_name, p.last_name, p.acuity_level,\n       vs.heart_rate, vs.spo2, vs.temp_f, vs.bp_systolic\nFROM fact_vital_signs vs\nJOIN dim_patients p ON vs.patient_id = p.patient_id\nWHERE vs.spo2 < 90 OR vs.heart_rate > 120 OR vs.temp_f > 101.5 OR vs.bp_systolic > 180\nORDER BY vs.timestamp DESC"""
        },
        {
            "question": "What medication overrides happened on high-alert drugs?",
            "query": """SELECT n.first_name, n.last_name,\n       m.name AS medication, m.drug_class,\n       ma.delay_minutes, ma.barcode_scanned\nFROM fact_medication_administration ma\nJOIN dim_nurses n ON ma.nurse_id = n.nurse_id\nJOIN dim_medications m ON ma.medication_id = m.medication_id\nWHERE ma.override_flag = true AND m.high_alert_flag = true\nORDER BY ma.actual_time DESC"""
        },
        {
            "question": "Which units have excessive overtime?",
            "query": """SELECT hu.unit_name, hu.unit_type,\n       COUNT(*) AS shifts_with_ot,\n       SUM(s.overtime_minutes) AS total_ot_min,\n       AVG(s.patient_count) AS avg_patients\nFROM fact_shifts s\nJOIN dim_hospital_units hu ON s.unit_id = hu.unit_id\nWHERE s.overtime_minutes > 60\nGROUP BY hu.unit_name, hu.unit_type\nORDER BY total_ot_min DESC"""
        },
        {
            "question": "Show handoffs with missed items.",
            "query": """SELECT n.first_name, n.last_name,\n       p.first_name AS patient_first, p.last_name AS patient_last,\n       hr.missed_items, hr.completeness_score\nFROM fact_handoff_reports hr\nJOIN dim_nurses n ON hr.from_nurse_id = n.nurse_id\nJOIN dim_patients p ON hr.patient_id = p.patient_id\nWHERE hr.missed_items > 2\nORDER BY hr.missed_items DESC"""
        },
        {
            "question": "Show after-hours documentation activity.",
            "query": """SELECT n.first_name, n.last_name,\n       dt.name AS doc_type,\n       de.duration_minutes, de.error_count\nFROM fact_documentation_events de\nJOIN dim_nurses n ON de.nurse_id = n.nurse_id\nJOIN dim_documentation_types dt ON de.doc_type_id = dt.doc_type_id\nWHERE de.after_hours_flag = true\nORDER BY de.start_time DESC"""
        },
        {
            "question": "Which documentation types have quality issues?",
            "query": """SELECT dt.name AS doc_type,\n       AVG(dq.completeness_score) AS avg_completeness,\n       AVG(dq.accuracy_score) AS avg_accuracy,\n       SUM(dq.audit_findings) AS total_findings\nFROM fact_documentation_quality dq\nJOIN dim_documentation_types dt ON dq.doc_type_id = dt.doc_type_id\nWHERE dq.completeness_score < 95 OR dq.accuracy_score < 95\nGROUP BY dt.name\nORDER BY avg_completeness ASC"""
        },
        {
            "question": "Show care plans needing review.",
            "query": """SELECT n.first_name, n.last_name,\n       p.first_name AS patient_first, d.name AS diagnosis,\n       cp.review_date, cp.status\nFROM fact_care_plans cp\nJOIN dim_nurses n ON cp.nurse_id = n.nurse_id\nJOIN dim_patients p ON cp.patient_id = p.patient_id\nJOIN dim_diagnoses d ON cp.diagnosis_id = d.diagnosis_id\nWHERE cp.review_date < GETDATE()\nORDER BY cp.review_date ASC"""
        },
        {
            "question": "Show patients with low satisfaction scores.",
            "query": """SELECT p.first_name, p.last_name,\n       ps.overall_score, ps.communication_score, ps.pain_management_score\nFROM fact_patient_satisfaction ps\nJOIN dim_patients p ON ps.patient_id = p.patient_id\nWHERE ps.overall_score < 5\nORDER BY ps.overall_score ASC"""
        },
        {
            "question": "Show EHR errors by module.",
            "query": """SELECT module,\n       COUNT(*) AS errors,\n       AVG(click_count) AS avg_clicks\nFROM fact_ehr_interactions\nWHERE error_flag = true\nGROUP BY module\nORDER BY errors DESC"""
        },
    ],

    EVENTHOUSE_DATABASE: [
        {
            "question": "Are there unacknowledged high-severity clinical alerts?",
            "query": """clinical_alerts\n| where severity == 'High' and acknowledged == false\n| project patient_id, unit_id, alert_type, response_time_sec"""
        },
        {
            "question": "Show BCMA scan mismatches and overrides.",
            "query": """bcma_scans\n| where scan_result != 'match' or override_reason != ''\n| project nurse_id, patient_id, scan_result, override_reason, timestamp\n| order by timestamp desc"""
        },
        {
            "question": "Which nurse calls have slow response?",
            "query": """nurse_call_events\n| where response_time_sec > 300\n| project patient_id, unit_id, call_type, response_time_sec\n| order by response_time_sec desc"""
        },
        {
            "question": "Show nurse location patterns.",
            "query": """rtls_location\n| where timestamp > ago(8h)\n| summarize total_dwell = sum(dwell_minutes) by nurse_id, unit_id, zone\n| order by total_dwell desc"""
        },
        {
            "question": "What is the EHR click activity today?",
            "query": """ehr_clickstream\n| where timestamp > ago(24h)\n| summarize clicks = count() by nurse_id, module, screen\n| order by clicks desc"""
        },
        {
            "question": "Show clinical alert trends.",
            "query": """clinical_alerts\n| where timestamp > ago(7d)\n| summarize count() by bin(timestamp, 1d), severity\n| order by timestamp asc"""
        },
        {
            "question": "What medication administrations happened in the last hour?",
            "query": """medication_administration\n| where timestamp > ago(1h)\n| project nurse_id, patient_id, medication_id, delay_minutes, override_flag\n| order by timestamp desc"""
        },
        {
            "question": "Show nurse call volume by unit.",
            "query": """nurse_call_events\n| where timestamp > ago(24h)\n| summarize calls = count(), avg_response = avg(response_time_sec) by unit_id, call_type\n| order by calls desc"""
        },
    ],
}

print(f"OPS_FEWSHOTS set: {', '.join(f'{k}: {len(v)} queries' for k, v in OPS_FEWSHOTS.items())}")

## Sample Questions for the Healthcare Data Agents

### QA Agent — `Healthcare_QA_Agent`

| # | Category | Sample Question |
|---|----------|------------------|
| 1 | Nurses | Which nurses have the highest documentation burden? |
| 2 | Nurses | Show nurses by unit and role. |
| 3 | Documentation | What is the average documentation time by document type? |
| 4 | Documentation | Show after-hours documentation rates. |
| 5 | Quality | Which nurses have documentation quality issues? |
| 6 | Quality | Show document completeness scores by type. |
| 7 | Patients | Show patient satisfaction by nurse. |
| 8 | Patients | Which patients have the highest acuity? |
| 9 | Medications | What is the medication administration delay rate? |
| 10 | Medications | How many medication overrides occurred? |
| 11 | Units | Which units have the most overtime? |
| 12 | Units | Show nurse-to-patient ratios by unit. |
| 13 | Handoffs | How many handoff reports have missed items? |
| 14 | Vitals | Show patient vital sign trends. |
| 15 | Care Plans | Show care plan status by diagnosis. |
| 16 | EHR | What is the EHR interaction volume by module? |
| 17 | Burnout | Which nurses are at burnout risk? |
| 18 | Assessments | What is the average assessment completeness? |
| 19 | Encounters | Show encounter types by unit. |
| 20 | Diagnoses | Show diagnoses by documentation complexity. |

### Ops Agent — `Healthcare_Ops_Agent`

| # | Category | Sample Question |
|---|----------|------------------|
| 1 | Burnout | Which nurses are at burnout risk? |
| 2 | Burnout | Who has the highest admin burden score? |
| 3 | Vitals | Are there any critical vital sign readings? |
| 4 | Medication | What medication overrides happened on high-alert drugs? |
| 5 | Medication | Show BCMA scan mismatches. |
| 6 | Alerts | Are there unacknowledged clinical alerts? |
| 7 | Calls | Which nurse calls have slow response times? |
| 8 | Overtime | Which units have excessive overtime? |
| 9 | Handoffs | Show handoffs with missed items. |
| 10 | Documentation | Show after-hours documentation activity. |
| 11 | Quality | Which documentation types have quality issues? |
| 12 | Care Plans | Show care plans needing review. |
| 13 | Satisfaction | Show patients with low satisfaction scores. |
| 14 | EHR | Show EHR errors by module. |
| 15 | Location | Where are nurses spending the most time? |
| 16 | Trends | Show clinical alert trends this week. |

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SAVE INSTRUCTIONS FOR PARENT NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

import json

_result = json.dumps({
    "qa": QA_AGENT_INSTRUCTIONS,
    "ops": OPS_AGENT_INSTRUCTIONS,
    "qa_fewshots": QA_FEWSHOTS,
    "ops_fewshots": OPS_FEWSHOTS,
    "qa_ds_instructions": QA_DS_INSTRUCTIONS,
    "ops_ds_instructions": OPS_DS_INSTRUCTIONS
})

_tmp_path = "file:/lakehouse/default/Files/_temp/agent_instructions.json"
notebookutils.fs.put(_tmp_path, _result, True)
print(f"  Written {len(_result):,} bytes to {_tmp_path}")

print(f"{'═'*60}")
print(f"AGENT INSTRUCTIONS LOADED — {INDUSTRY.upper()}")
print(f"{'═'*60}")
print(f"")
print(f"  QA_AGENT_INSTRUCTIONS:  {len(QA_AGENT_INSTRUCTIONS):,} chars")
print(f"  OPS_AGENT_INSTRUCTIONS: {len(OPS_AGENT_INSTRUCTIONS):,} chars")
print(f"  QA_FEWSHOTS:  {sum(len(v) for v in QA_FEWSHOTS.values())} total queries across {len(QA_FEWSHOTS)} datasources")
print(f"  OPS_FEWSHOTS: {sum(len(v) for v in OPS_FEWSHOTS.values())} total queries across {len(OPS_FEWSHOTS)} datasources")

notebookutils.notebook.exit("ok")